# OOS Detection — embedder sensitivity (AutoIntent few-shot)

**Гипотеза:** качество OOS-детекции чувствительно к выбору fixed embedder.

Сравниваем (few-shot, `classic-light`, seeds 42/123/456, shots 10/20/50):
- **stella** — `dunzhang/stella_en_400M_v5`
- **e5_large** — `intfloat/multilingual-e5-large` (без `-instruct`)

Full-train прогоны **отключены**. Артефакты: `/kaggle/working`.

In [ ]:
# 1. Setup
import os, sys, subprocess

for _k, _v in {
    "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
    "NUMEXPR_NUM_THREADS": "1", "TOKENIZERS_PARALLELISM": "false",
}.items():
    os.environ.setdefault(_k, _v)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "autointent", "datasets>=2.14", "sentence-transformers>=2.7",
    "transformers>=4.40", "scikit-learn>=1.3", "pandas"], check=False)
# Stella custom attention needs xformers on Kaggle
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xformers"], check=False)

import gc, json, time, random, traceback, platform
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
)

def _patch_trust_remote_code():
    import sys as _sys
    import transformers.dynamic_module_utils as _dmu
    _dmu.resolve_trust_remote_code = lambda *a, **kw: True
    for _n, _m in list(_sys.modules.items()):
        if _n.startswith("transformers.") and _m is not None:
            if getattr(_m, "resolve_trust_remote_code", None):
                _m.resolve_trust_remote_code = lambda *a, **kw: True

_patch_trust_remote_code()

CUDA_OK = torch.cuda.is_available()
print(f"cuda={CUDA_OK} device={torch.cuda.get_device_name(0) if CUDA_OK else 'cpu'}")

In [ ]:
# 2. Config
HF_TOKEN = ""
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        for _k in ("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN"):
            try:
                HF_TOKEN = UserSecretsClient().get_secret(_k)
                if HF_TOKEN: break
            except Exception:
                pass
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception:
        pass

SEED_GLOBAL = 42
SEEDS = [42, 123, 456]
SHOTS = [10, 20, 50]
PRESET = "classic-light"
OOS_LABEL = -1

# Embedders under comparison (fixed, no AutoML embedder search)
EMBEDDERS = {
    "stella": "dunzhang/stella_en_400M_v5",
    "e5_large": "intfloat/multilingual-e5-large",
}

WORKDIR = Path("/kaggle/working")
if not WORKDIR.exists():
    WORKDIR = Path.cwd() / "kaggle_working"
WORKDIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = WORKDIR / "runs"
RUNS_DIR.mkdir(exist_ok=True)

METRICS_JSON = WORKDIR / "embedder_sensitivity_metrics.json"
# локальная копия в репо (если ноутбук не на Kaggle)
_REPO_RESULTS = Path("results") if (Path.cwd() / "results").exists() else Path(__file__).parent.parent / "results" if False else None
METRICS_JSON_REPO = (Path.cwd().parent / "results" / "embedder_sensitivity_metrics.json"
                     if (Path.cwd().name == "notebooks" and (Path.cwd().parent / "results").exists())
                     else None)
SUMMARY_JSON = WORKDIR / "embedder_sensitivity_summary.json"
RESULTS_CSV = WORKDIR / "embedder_sensitivity_results.csv"
FAILED_LOG = WORKDIR / "embedder_sensitivity_failed.jsonl"

random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)
if CUDA_OK:
    torch.cuda.manual_seed_all(SEED_GLOBAL)

print("embedders:", EMBEDDERS)
print("working:", WORKDIR)

In [ ]:
# 3. Load CLINC OOS (clinc_oos plus)
from datasets import load_dataset

def _load_clinc():
    kw = {"token": HF_TOKEN} if HF_TOKEN else {}
    for name, cfg in [("clinc_oos", "plus"), ("DeepPavlov/clinc150", None)]:
        try:
            ds = load_dataset(name, cfg, **kw) if cfg else load_dataset(name, **kw)
            print(f"dataset: {name}{'/' + cfg if cfg else ''}")
            return ds
        except Exception as e:
            print(f"  skip {name}: {e}")
    raise RuntimeError("Could not load CLINC OOS dataset")

ds = _load_clinc()
print("splits:", {k: len(v) for k, v in ds.items()})

In [ ]:
# 4. Prepare labels (OOS = -1, in-scope = 0..N-1)
_split = "train" if "train" in ds else list(ds.keys())[0]
_testk = "test" if "test" in ds else "validation"
cols = list(ds[_split].column_names)
TEXT_COL = next((c for c in ("text", "utterance", "sentence") if c in cols), cols[0])
LABEL_COL = next((c for c in ("intent", "label") if c in cols), cols[1])

label_names = list(ds[_split].features[LABEL_COL].names)
OOS_RAW = next((i for i, n in enumerate(label_names) if n.lower() in ("oos", "out_of_scope", "out-of-scope")), len(label_names) - 1)

in_scope_ids = sorted({r[LABEL_COL] for r in ds[_split] if r[LABEL_COL] != OOS_RAW}
                      | {r[LABEL_COL] for r in ds[_testk] if r[LABEL_COL] != OOS_RAW})
RAW2DENSE = {old: i for i, old in enumerate(in_scope_ids)}
ID2NAME = {RAW2DENSE[o]: label_names[o] for o in in_scope_ids}
N_CLASSES = len(in_scope_ids)

def _remap(raw):
    return OOS_LABEL if raw == OOS_RAW else RAW2DENSE[raw]

def _extract(split_name):
    texts = [r[TEXT_COL] for r in ds[split_name]]
    labels = [_remap(r[LABEL_COL]) for r in ds[split_name]]
    return texts, labels

train_texts, train_labels = _extract(_split)
test_texts, test_labels = _extract(_testk)

assert len(train_texts) == len(train_labels) == len([t for t in train_texts if t.strip()])
assert len(test_texts) == len(test_labels)

def _stats(labels):
    a = np.asarray(labels)
    return {"n": len(a), "oos": int((a == OOS_LABEL).sum()), "inscope": int((a != OOS_LABEL).sum())}

DATA_INFO = {
    "train": _stats(train_labels),
    "test": _stats(test_labels),
    "n_classes": N_CLASSES,
    "oos_label_name": label_names[OOS_RAW],
}
print("data:", DATA_INFO)

In [ ]:
# 5. Metrics + few-shot sampler
def sample_fewshot(n_shots, seed):
    rng = np.random.default_rng(seed)
    by_lab = {}
    for t, l in zip(train_texts, train_labels):
        by_lab.setdefault(l, []).append(t)
    sel_t, sel_l = [], []
    n_int = sum(1 for k in by_lab if k != OOS_LABEL)
    for lab, pool in by_lab.items():
        if lab == OOS_LABEL:
            n_take = min(n_shots * max(1, n_int), len(pool))
        else:
            n_take = min(n_shots, len(pool))
        for i in rng.choice(len(pool), size=n_take, replace=False):
            sel_t.append(pool[i]); sel_l.append(lab)
    return sel_t, sel_l

def compute_metrics(y_true, y_pred, y_scores=None):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    yo_t = (y_true == OOS_LABEL).astype(int)
    yo_p = (y_pred == OOS_LABEL).astype(int)
    tp = int(((y_true == OOS_LABEL) & (y_pred == OOS_LABEL)).sum())
    fp = int(((y_true != OOS_LABEL) & (y_pred == OOS_LABEL)).sum())
    fn = int(((y_true == OOS_LABEL) & (y_pred != OOS_LABEL)).sum())
    tn = int(((y_true != OOS_LABEL) & (y_pred != OOS_LABEL)).sum())
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_oos": float(f1_score(yo_t, yo_p, zero_division=0)),
        "oos_precision": float(precision_score(yo_t, yo_p, zero_division=0)),
        "oos_recall": float(recall_score(yo_t, yo_p, zero_division=0)),
        "oos_fpr": float(fp / max(fp + tn, 1)),
        "tp_oos": tp, "fp_oos": fp, "fn_oos": fn, "tn_oos": tn,
        "n_eval": len(y_true),
        "n_oos_true": int((y_true == OOS_LABEL).sum()),
        "n_inscope_true": int((y_true != OOS_LABEL).sum()),
        "pred_oos": int((y_pred == OOS_LABEL).sum()),
    }
    out["auroc_oos"] = out["auprc_oos"] = None
    if y_scores is not None and len(np.unique(yo_t)) == 2:
        try:
            out["auroc_oos"] = float(roc_auc_score(yo_t, y_scores))
            out["auprc_oos"] = float(average_precision_score(yo_t, y_scores))
        except Exception:
            pass
    return out

In [ ]:
# 6. Run one experiment
from autointent import Pipeline, Dataset as AIDataset
from autointent.configs import LoggingConfig, EmbedderConfig, DataConfig

def _to_ai(texts, labels):
    out = []
    for t, l in zip(texts, labels):
        out.append({"utterance": t} if l == OOS_LABEL else {"utterance": t, "label": int(l)})
    return out

def _free():
    gc.collect()
    if CUDA_OK:
        torch.cuda.empty_cache()

_loaded_embedders = set()

def run_experiment(embedder_key, hf_model, n_shots, seed):
    run_id = f"oos_{PRESET}_{embedder_key}_{n_shots}shot_seed{seed}"
    run_dir = RUNS_DIR / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    print(f"RUN START embedder={embedder_key} shots={n_shots} seed={seed}")

    tr_t, tr_l = sample_fewshot(n_shots, seed)
    intents = [{"id": i, "name": ID2NAME[i]} for i in sorted({l for l in tr_l if l != OOS_LABEL})]
    ai_ds = AIDataset.from_dict({
        "train": _to_ai(tr_t, tr_l),
        "test": _to_ai(test_texts, test_labels),
        "intents": intents,
    })

    pipeline = Pipeline.from_preset(PRESET)
    pipeline.set_config(EmbedderConfig(model_name=hf_model))
    pipeline.set_config(DataConfig(scheme="cv", n_folds=3))
    pipeline.set_config(LoggingConfig(project_dir=run_dir, dump_modules=True, clear_ram=False))

    t0 = time.perf_counter()
    pipeline.fit(ai_ds)
    train_sec = time.perf_counter() - t0
    pipeline.dump(run_dir)
    _loaded_embedders.add(embedder_key)

    e0 = time.perf_counter()
    raw = pipeline.predict(test_texts)
    eval_sec = time.perf_counter() - e0
    y_pred = np.array([OOS_LABEL if p is None else int(p) for p in raw])

    y_scores = None
    try:
        if hasattr(pipeline, "predict_with_metadata"):
            res = pipeline.predict_with_metadata(test_texts)
            arrs = [u.score for u in (getattr(res, "utterances", None) or []) if getattr(u, "score", None) is not None]
            if arrs:
                a = np.asarray(arrs)
                y_scores = 1.0 - a.max(axis=1) if a.ndim == 2 else None
    except Exception:
        pass

    m = compute_metrics(test_labels, y_pred, y_scores)
    rec = {
        "task": "oos_detection",
        "hypothesis": "embedder_sensitivity",
        "embedder_key": embedder_key,
        "embedder_hf_model": hf_model,
        "preset": PRESET,
        "mode": f"{n_shots}shot",
        "n_shots": n_shots,
        "seed": seed,
        "train_n": len(tr_t),
        "train_sec": round(train_sec, 1),
        "eval_sec": round(eval_sec, 2),
        "status": "ok",
        **m,
    }
    (run_dir / "metrics.json").write_text(json.dumps(rec, indent=2))
    (WORKDIR / f"metrics_{run_id}.json").write_text(json.dumps(rec, indent=2))

    print(
        f"RUN OK embedder={embedder_key} shots={n_shots} seed={seed} "
        f"acc={m['accuracy']:.4f} macro_f1={m['macro_f1']:.4f} "
        f"f1_oos={m['f1_oos']:.4f} oos_rec={m['oos_recall']:.4f} "
        f"oos_prec={m['oos_precision']:.4f} auroc={m['auroc_oos']} "
        f"train_sec={train_sec:.0f}"
    )
    del pipeline
    _free()
    return rec

In [ ]:
# 7. Grid: 2 embedders x 3 shots x 3 seeds = 18 runs
ALL_RESULTS = []

for emb_key, hf_id in EMBEDDERS.items():
    for n_shots in SHOTS:
        for seed in SEEDS:
            stub = {"embedder_key": emb_key, "embedder_hf_model": hf_id,
                    "mode": f"{n_shots}shot", "n_shots": n_shots, "seed": seed, "preset": PRESET}
            try:
                rec = run_experiment(emb_key, hf_id, n_shots, seed)
            except Exception as e:
                rec = {**stub, "status": "failed",
                         "error_type": type(e).__name__, "error_message": str(e),
                         "traceback_short": "\n".join(traceback.format_exc().splitlines()[-8:]),
                         "timestamp": datetime.now(timezone.utc).isoformat()}
                with open(FAILED_LOG, "a") as f:
                    f.write(json.dumps(rec) + "\n")
                print(f"RUN FAILED embedder={emb_key} shots={n_shots} seed={seed} error={type(e).__name__}: {e}")
                _free()
            ALL_RESULTS.append(rec)

In [ ]:
# 8. Summary for hypothesis check
COLS = [
    "embedder_key", "n_shots", "seed", "status",
    "accuracy", "macro_f1", "weighted_f1",
    "f1_oos", "oos_precision", "oos_recall", "oos_fpr",
    "auroc_oos", "auprc_oos",
    "tp_oos", "fp_oos", "fn_oos", "tn_oos",
    "train_sec", "eval_sec",
]

df = pd.DataFrame(ALL_RESULTS)
for c in COLS:
    if c not in df.columns:
        df[c] = None
df = df[COLS + [c for c in df.columns if c not in COLS]]
df.to_csv(RESULTS_CSV, index=False)
METRICS_JSON.write_text(json.dumps(ALL_RESULTS, indent=2, default=str))

# Mean per embedder x shots (aggregate over seeds)
ok = df[df["status"] == "ok"].copy()
agg = ok.groupby(["embedder_key", "n_shots"], as_index=False).agg({
    "accuracy": "mean", "macro_f1": "mean", "f1_oos": "mean",
    "oos_recall": "mean", "oos_precision": "mean", "auroc_oos": "mean",
    "train_sec": "mean",
})

summary = {
    "hypothesis": "OOS quality sensitive to fixed embedder (Stella vs e5-large)",
    "n_runs": len(df), "n_ok": int((df["status"] == "ok").sum()),
    "n_failed": int((df["status"] != "ok").sum()),
    "embedders": EMBEDDERS,
    "data": DATA_INFO,
    "mean_by_embedder_shots": agg.to_dict(orient="records"),
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2, default=str))

print("\n=== ALL RUNS (compact) ===")
print(df[["embedder_key", "n_shots", "seed", "status",
          "accuracy", "macro_f1", "f1_oos", "oos_recall", "oos_precision",
          "auroc_oos", "train_sec"]].to_string(index=False))

print("\n=== MEAN BY EMBEDDER x SHOTS (seeds pooled) ===")
if len(agg):
    print(agg.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
else:
    print("(no successful runs)")

print("\nSaved:", RESULTS_CSV.name, METRICS_JSON.name, SUMMARY_JSON.name)

In [ ]:
# 9. (optional) Append full + auto-embedder baseline runs to METRICS_JSON
# Запускать после few-shot grid, если full-прогоны делались отдельно.
FULL_AUTO_BASELINE = [
    {"seed": 42, "f1_oos": 0.8409, "oos_recall": 0.8350, "oos_precision": 0.8469,
     "auroc_oos": 0.9736906666666667, "train_sec": 242.3, "eval_sec": 0.1},
    {"seed": 123, "f1_oos": 0.8409, "oos_recall": 0.8350, "oos_precision": 0.8469,
     "auroc_oos": 0.9736906666666667, "train_sec": 171.3, "eval_sec": 0.0},
    {"seed": 456, "f1_oos": 0.8409, "oos_recall": 0.8350, "oos_precision": 0.8469,
     "auroc_oos": 0.9736906666666667, "train_sec": 170.0, "eval_sec": 0.0},
]

def _full_auto_record(r):
    return {
        "task": "oos_detection", "hypothesis": "embedder_sensitivity",
        "embedder_key": "auto", "embedder_hf_model": None, "preset": PRESET,
        "mode": "full", "n_shots": None, "seed": r["seed"],
        "train_n": None, "train_sec": r["train_sec"], "eval_sec": r["eval_sec"],
        "status": "ok",
        "accuracy": None, "macro_f1": None, "weighted_f1": None,
        "oos_fpr": None, "auprc_oos": None,
        "tp_oos": None, "fp_oos": None, "fn_oos": None, "tn_oos": None,
        "n_eval": None, "n_oos_true": None, "n_inscope_true": None,
        "pred_oos": None, "pred_inscope": None,
        "f1_oos": r["f1_oos"], "oos_recall": r["oos_recall"],
        "oos_precision": r["oos_precision"], "auroc_oos": r["auroc_oos"],
    }

if METRICS_JSON.exists():
    _all = json.loads(METRICS_JSON.read_text())
else:
    _all = list(ALL_RESULTS) if "ALL_RESULTS" in dir() else []

_keys = {(x.get("embedder_key"), x.get("mode"), x.get("n_shots"), x.get("seed")) for x in _all}
for _r in FULL_AUTO_BASELINE:
    _rec = _full_auto_record(_r)
    _k = (_rec["embedder_key"], _rec["mode"], _rec["n_shots"], _rec["seed"])
    _all = [_rec if (y.get("embedder_key"), y.get("mode"), y.get("n_shots"), y.get("seed")) == _k else y for y in _all] if _k in _keys else _all + [_rec]
    if _k not in _keys:
        _keys.add(_k)
        _all.append(_rec)

METRICS_JSON.write_text(json.dumps(_all, indent=2, ensure_ascii=False))
print(f"Merged full/auto into {METRICS_JSON} ({len(_all)} records)")
